<a href="https://colab.research.google.com/github/vishal-8433code/car-price-prediction/blob/main/vehicle_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [67]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.metrics import r2_score,mean_squared_error,accuracy_score

In [68]:
car_data = pd.read_csv("/content/car data.csv")
car_detail = pd.read_csv("/content/CAR DETAILS FROM CAR DEKHO.csv")
car_data_v3 = pd.read_csv("/content/Car details v3.csv")
car_data_v4 = pd.read_csv("/content/car details v4.csv")


In [69]:
car_data["Fuel_Type"] = car_data['Fuel_Type'].astype("category")
car_data["Seller_Type"] = car_data['Seller_Type'].astype("category")
car_data["Transmission"] = car_data['Transmission'].astype("category")
car_data["Owner"] = car_data['Owner'].astype("category")

In [70]:
def drop_wrong_values(value):
    data = car_data[car_data["Car_Name"].str.contains(value)].index
    car_data.drop(index = data,inplace = True)
value = ["TVS","Mahindra","Hyosung","UM","Royal","KTM","Bajaj","Honda","Yamaha","Hero","Activa","Access"]
for i in range(len(value)):
  drop_wrong_values(value[i])

In [71]:
## remove outlier
def remove_outlier(column):
    q1 = car_data[column].quantile(0.25)
    q3 = car_data[column].quantile(0.75)

    iqr = q3 - q1
    upper_limit = q3 + 1.5*iqr
    lower_limit = q1 - 1.5*iqr
    car_data.drop(index= car_data[(car_data[column] >= upper_limit) | (car_data[column] <= lower_limit)].index,inplace=True)

columns = ["Selling_Price","Present_Price","Kms_Driven"]
for i in range(len(columns)):
  remove_outlier(columns[i])



In [72]:
car_data["age"] = 2026 - car_data["Year"]
car_data.drop(columns=["Year","Car_Name"],inplace=True)

In [73]:
## remove outlier
car_data.info()
car_data.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 169 entries, 0 to 300
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Selling_Price  169 non-null    float64 
 1   Present_Price  169 non-null    float64 
 2   Kms_Driven     169 non-null    int64   
 3   Fuel_Type      169 non-null    category
 4   Seller_Type    169 non-null    category
 5   Transmission   169 non-null    category
 6   Owner          169 non-null    category
 7   age            169 non-null    int64   
dtypes: category(4), float64(2), int64(2)
memory usage: 7.8 KB


,Selling_Price,Present_Price,Kms_Driven,age
count,169.000000,169.000000,169.000000,169.000000
mean,5.428225,8.333314,37303.035503,12.159763
std,2.488134,3.030737,19923.136766,2.564277
min,1.050000,2.690000,2071.000000,8.000000
25%,3.500000,5.900000,21200.000000,10.000000
50%,4.950000,7.710000,36198.000000,11.000000
75%,6.700000,9.900000,50000.000000,13.000000
max,12.900000,16.090000,90000.000000,23.000000


In [74]:
x = car_data.drop(columns=["Selling_Price"])
y = car_data["Selling_Price"]
x.head()

,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,age
0,5.59,27000,Petrol,Dealer,Manual,0,12
1,9.54,43000,Diesel,Dealer,Manual,0,13
2,9.85,6900,Petrol,Dealer,Manual,0,9
3,4.15,5200,Petrol,Dealer,Manual,0,15
4,6.87,42450,Diesel,Dealer,Manual,0,12


In [75]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)

In [76]:
trf = ColumnTransformer(transformers=[
    ("trf1",OneHotEncoder(sparse_output=False,handle_unknown="ignore"),["Fuel_Type","Seller_Type","Transmission"]),
    ("trf3",StandardScaler(),["Present_Price","Kms_Driven","age"])
],remainder="passthrough")

pipe = Pipeline([
   ("preprocessing",trf),
   ("model",LinearRegression())
])

pipe.fit(x_train,y_train)
y_pred = pipe.predict(x_test)
print(r2_score(y_test,y_pred))
print(mean_squared_error(y_test,y_pred))
print(np.sqrt(mean_squared_error(y_test,y_pred)))
print("Train R2:", pipe.score(x_train, y_train))
print("Test R2 :", pipe.score(x_test, y_test))




0.8813330822474442
0.6394478138995121
0.799654809214271
Train R2: 0.8570816318261637
Test R2 : 0.8813330822474442
